In [9]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

response = llm.invoke("Hello")
print(response.content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPXvHq7haeN9LuDG5MbTkuGaSTaxfz5LNHn2Vgl0jnxpAZaqvIF+syAfVX7vrFwQjtqxj3BFq2j5oFaxulNNilc22TZvmKm9wml0XQ/4h2zk8encPHxO6d'}}]


In [ ]:
# For streaming
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    if chunk.content:
        print(chunk.content[0]["text"], end="", flush=True)

Silent code awakes to think and roam,
They steer the digital world they call home.

## Message Passing into Langchain

In [14]:
messages = [
    SystemMessage("You are an assistant that answers in exactly five words"),
    HumanMessage("What is the capital of France")
]

print(llm.invoke(messages).content[0]["text"])

The capital is Paris now.


## Tools with @tools decorator

In [ ]:
@tool
def get_share_price(symbol:str) ->float:
    """Return the current share price for a given ticker symbol"""
    fake_prices =  {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print(get_share_price.name)
print(get_share_price.description)

get_share_price
Return the current share price for a given ticker symbol


In [ ]:
llm_with_tools = llm.bind_tools([get_share_price])
response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content: ", repr(response.content))  # nothing comes out from this
print("tool_calls: ", response.tool_calls)

content:  []
tool_calls:  [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_363176', 'type': 'tool_call'}]


In [ ]:
# Running the tool loop wise
conversation = [HumanMessage("What is share price of Amazon")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls: # see the tools and if the given tool is there run it twice
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
print(llm_with_tools.invoke(messages).content[0]["text"])

[{'type': 'text', 'text': 'The share price of Amazon (AMZN) is $198.', 'extras': {'signature': 'El4KXAERTTIPhfjfOopsrIRanndAQAwYR/5G1lpWj01TrCpBY4y8Znegly66JrUzDAKXTAq0MBQbxrZ7spBxOLuvWrKiO6+AlbInbLWzZKyLUY8TWVuWq9vHCHKwIwfU'}}]


## Structured output with langchain

In [ ]:
class Company(BaseModel):
    name:str = Field(description="The company name")
    ticker:str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year that it was founded in")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon company")
print(company)
print("Ticker: ", company.ticker)

name='Amazon.com, Inc.' ticker='AMZN' founded_year=1994
Ticker:  AMZN
